In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Raghav\\Desktop\\imcaption\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Raghav\\Desktop\\imcaption'

In [5]:
import mlflow
import mlflow.keras
from urllib.parse import urlparse

c:\Users\Raghav\.conda\envs\imgcaption\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import dagshub
dagshub.init(repo_owner='divyanshrajput118', repo_name='imcaption', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

c:\Users\Raghav\.conda\envs\imgcaption\Lib\site-packages\rich\live.py:229: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=4057bd9c-9871-428b-93fc-8d9c59aec431&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=8cc2c4931855a0c660f03d576a33828397e439f6cc544f2563b833e0b082fd91




Accessing as divyanshrajput118

Initialized MLflow to track repo "divyanshrajput118/imcaption"

Repository divyanshrajput118/imcaption initialized!

In [7]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    vectorizer_path: Path
    images_dir: Path
    test_images_captions_path: Path
    image_feex_path: Path
    trained_model_path: Path
    scores_path: Path
    all_params: dict
    mlflow_uri: str 

    MAX_LENGTH: int
    BEAM_WIDTH: int

In [8]:
from imgCaption.constants import *
from imgCaption.utils.common import read_yaml,create_directories,save_json,load_pkl_file

In [9]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> ModelEvaluationConfig:
        eval_cfg = self.config.model_evaluation
        training = self.config.model_trainer
        data_transformation = self.config.data_transformation
        prepare_base_model = self.config.prepare_base_model

        create_directories([eval_cfg.root_dir])

        evaluation_config = ModelEvaluationConfig(
                                                    root_dir=Path(eval_cfg.root_dir),
                                                    vectorizer_path=Path(data_transformation.vectorizer_path),
                                                    images_dir=Path(training.images_dir),
                                                    test_images_captions_path=Path(data_transformation.test_images_captions_path),
                                                    image_feex_path=Path(prepare_base_model.image_feex_path),
                                                    trained_model_path=Path(training.trained_model_path),
                                                    scores_path=Path(eval_cfg.scores_path),
                                                    all_params=self.params,
                                                    mlflow_uri=eval_cfg.mlflow_uri,
                                                    MAX_LENGTH=self.params.MAX_LENGTH,
                                                    BEAM_WIDTH=self.params.BEAM_WIDTH,
                                                )

        return evaluation_config

In [11]:
import gc
import json
import numpy as np
from pathlib import Path
from tqdm import tqdm
from imgCaption import logger
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.sequence import pad_sequences
from nltk.translate.bleu_score import corpus_bleu

In [12]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
        self.load_feature_extractor()
        self.load_trained_model()
        self.load_vectorizer()

    def load_feature_extractor(self):
        self.feature_extractor = load_model(self.config.image_feex_path)

    def load_trained_model(self):
        self.trained_model = load_model(self.config.trained_model_path)
    
    def load_vectorizer(self):
        from_disk = load_pkl_file(self.config.vectorizer_path)
        self.vectorizer = TextVectorization.from_config(from_disk["config"])

        if "vocabulary" in from_disk and len(from_disk["vocabulary"]) > 2:
            vocab = [w for w in from_disk["vocabulary"] if w not in ('', '[UNK]')]
            self.vectorizer.set_vocabulary(vocab)
            logger.info(f"Vectorizer loaded via vocabulary list. Size: {len(self.vectorizer.get_vocabulary())}")
        elif from_disk.get("weights"):
            self.vectorizer.adapt(tf.data.Dataset.from_tensor_slices(["dummy"]))
            self.vectorizer.set_weights(from_disk["weights"])
            logger.info(f"Vectorizer loaded via set_weights. Size: {len(self.vectorizer.get_vocabulary())}")
        else:
            raise ValueError(
                "Could not restore vectorizer vocabulary. "
                "Re-run Stage 2 (data_transformation) to regenerate the vectorizer_data.pkl with vocabulary saved."
            )
        self.vocab = self.vectorizer.get_vocabulary()
        self.word_to_index = {w: i for i, w in enumerate(self.vocab)}

    def features_ext_dict(self,image_caption_map):
        d={}
        images=list(image_caption_map.keys())
        batch_size = 32

        for start in tqdm(range(0, len(images), batch_size), desc="Extracting features", unit="batch"):
            batch_names = images[start : start + batch_size]
            batch_imgs = []
            for img_name in batch_names:
                img_path = os.path.join(self.config.images_dir, img_name)
                img = load_img(img_path, target_size=(224, 224))
                img = img_to_array(img)
                img = preprocess_input(img)
                batch_imgs.append(img) 
            batch_imgs = np.array(batch_imgs)
            features = self.feature_extractor.predict(batch_imgs, verbose=0)
            for i, image in enumerate(batch_names):
                d[image] = features[i].flatten()                
            del batch_imgs, features
        logger.info(f"Feature extraction complete: {len(d)} images processed")
        return d
    
    def encode_sentence(self, sentence: str) -> np.ndarray:
        token_ids = self.vectorizer([sentence]).numpy()[0]
        padded = pad_sequences([token_ids], maxlen=self.config.MAX_LENGTH, padding='post')
        return padded

    def decode_index(self, idx: int) -> str | None:
        if idx <= 1 or idx >= len(self.vocab):
            return None
        return self.vocab[idx]

    def save_score(self, results: dict):
        save_json(path=self.config.scores_path, data=results)
        logger.info(f"Scores saved to {self.config.scores_path}")

    def generate_caption(self, image_feature: np.ndarray | None = None, beam_width: int | None = None) -> str:
        beam_width = beam_width or self.config.BEAM_WIDTH
        max_length = self.config.MAX_LENGTH

        img_vec = np.reshape(image_feature, (1, 2048))

        sequences = [(['startseq'], 0.0)]

        for _ in range(max_length):
            all_candidates = []

            for caption_words, score in sequences:
                if caption_words[-1] == 'endseq':
                    all_candidates.append((caption_words, score))
                    continue

                current_sentence = " ".join(caption_words)
                seq = self.encode_sentence(current_sentence)

                yhat = self.trained_model(
                    [img_vec, seq], training=False
                ).numpy()[0]

                yhat[0] = 0.0
                top_indices = np.argsort(yhat)[-beam_width:]

                for idx in top_indices:
                    word = self.decode_index(int(idx))
                    if word is None:
                        continue
                    prob = float(yhat[idx])
                    new_score = score + np.log(prob + 1e-10)
                    all_candidates.append((caption_words + [word], new_score))

            ordered = sorted(all_candidates, key=lambda x: x[1], reverse=True)
            sequences = ordered[:beam_width]

            if all(seq[-1] == 'endseq' for seq, _ in sequences):
                break

        best_words, _ = max(sequences, key=lambda x: x[1] / max(1, len(x[0]) - 1))

        result = [w for w in best_words if w not in ('startseq', 'endseq')]
        return " ".join(result).strip()

    def evaluate_bleu(
        self,
        test_features: dict,
        test_captions: dict,
        checkpoint_every: int = 100,
    ) -> dict:
        actual, predicted = [], []
        start_idx = 0

        ckpt_path = self.config.root_dir / "eval_checkpoint.json"
        if ckpt_path.exists():
            ckpt = json.loads(ckpt_path.read_text())
            actual    = ckpt["actual"]
            predicted = [p.split() for p in ckpt["predicted_str"]]
            start_idx = len(actual)
            logger.info(f"Resuming from checkpoint at {start_idx}/{len(test_captions)}")

        img_ids = list(test_captions.keys())

        for i in tqdm(range(start_idx, len(img_ids)), total=len(img_ids), initial=start_idx):
            img_id = img_ids[i]

            references = []
            for cap in test_captions[img_id]:
                words = [
                    w for w in cap.split()
                    if w not in ('startseq', 'endseq')
                ]
                if words:
                    references.append(words)
            actual.append(references)

            pred_caption = self.generate_caption(
                test_features[img_id],
                beam_width=self.config.BEAM_WIDTH,
            )
            predicted.append(pred_caption.split())

            if (i + 1) % checkpoint_every == 0:
                gc.collect()
                ckpt_path.write_text(json.dumps({
                    "actual": actual,
                    "predicted_str": [" ".join(p) for p in predicted],
                }))
                logger.info(f"Checkpoint saved at {i + 1}/{len(img_ids)}")

        bleu1 = corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0))
        bleu2 = corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0))
        bleu3 = corpus_bleu(actual, predicted, weights=(0.33, 0.33, 0.33, 0))
        bleu4 = corpus_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25))

        logger.info(f"BLEU-1: {bleu1:.4f}")
        logger.info(f"BLEU-2: {bleu2:.4f}")
        logger.info(f"BLEU-3: {bleu3:.4f}")
        logger.info(f"BLEU-4: {bleu4:.4f}")

        results = {
            "bleu1": round(bleu1, 6),
            "bleu2": round(bleu2, 6),
            "bleu3": round(bleu3, 6),
            "bleu4": round(bleu4, 6),
        }

        self.save_score(results)

        if ckpt_path.exists():
            ckpt_path.unlink()

        return results

    def evaluate(self):
        logger.info("Loading test captions...")
        test_captions = load_pkl_file(self.config.test_images_captions_path)
        logger.info(f"Test images: {len(test_captions)}")

        logger.info("Extracting image features for test set...")
        test_features = self.features_ext_dict(test_captions)

        logger.info("Running BLEU evaluation...")
        results = self.evaluate_bleu(test_features, test_captions)
        self.log_into_mlflow(results) 

        return results
    
    def log_into_mlflow(self, results: dict):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            mlflow.log_params(dict(self.config.all_params))
            mlflow.log_metrics(results)   

        
            if tracking_url_type_store != "file":
                mlflow.keras.log_model(
                    self.trained_model, "model",
                    registered_model_name="ImageCaptionModel"
                )
            else:
                mlflow.keras.log_model(self.trained_model, "model")

    


In [ ]:
config = ConfigurationManager()
model_evaluation_config = config.get_evaluation_config()
model_evaluation = ModelEvaluation(config=model_evaluation_config)
results = model_evaluation.evaluate()

logger.info(f"Final evaluation results: {results}")

[2026-07-09 21:21:45,294: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-09 21:21:45,313: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-09 21:21:45,320: INFO: common: created directory at: artifacts]
[2026-07-09 21:21:45,323: INFO: common: created directory at: artifacts/model_evaluation]
[2026-07-09 21:21:49,127: WARNING: config: TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.]
[2026-07-09 21:21:50,821: INFO: common: Pickle file loaded from: artifacts\data_transformation\vectorizer_data.pkl]
[2026-07-09 21:21:51,083: INFO: 1255752416: Vectorizer loaded via vocabulary list. Size: 8782]
[2026-07-09 21:21:51,128: INFO: 1255752416: Loading test captions...]
[2026-07-09 21:21:51,175: INFO: common: Pickle file loaded from: artifacts\data_transformation\test_images_captions.pkl]
[2026-07-09 21:21:51,177: I

Extracting features: 100%|██████████| 51/51 [02:29<00:00,  2.93s/batch]

[2026-07-09 21:24:20,677: INFO: 1255752416: Feature extraction complete: 1619 images processed]
[2026-07-09 21:24:20,677: INFO: 1255752416: Running BLEU evaluation...]



  6%|▌         | 99/1619 [30:29<8:00:13, 18.96s/it] 

[2026-07-09 21:55:08,099: INFO: 1255752416: Checkpoint saved at 100/1619]


 10%|▉         | 159/1619 [49:14<6:33:42, 16.18s/it] 

In [ ]:
# try:
#     config = ConfigurationManager()
#     model_evaluation_config = config.get_evaluation_config()
#     model_evaluation = ModelEvaluation(config=model_evaluation_config)
#     test_features = model_evaluation.img_feature_extraction(model_evaluation_config.test_img_id_path)
#     caption = model_evaluation.evaluate_bleu(test_features, test_imagesid_captions, max_length=35, beam_width=3)
# except Exception as e:
#     raise e